In [ ]:
import os
import sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
import torch
from copy import deepcopy
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('scaling_notebook')

# Add project root to Python path
current_dir = os.getcwd()
if not any(p.endswith('dendritic_modeling') for p in sys.path):
    if os.path.basename(current_dir) == 'notebooks':
        project_root = os.path.dirname(current_dir)
    else:
        project_root = current_dir
    sys.path.insert(0, project_root)
    logger.info(f"Added {project_root} to Python path")

# Import main training function
try:
    from dendritic_modeling.train_experiments import main as train_main
    logger.info("Successfully imported train_main from dendritic_modeling")
except ImportError as e:
    logger.error(f"Failed to import: {e}")
    raise

In [ ]:
def load_config_yaml(path):
    """Load a YAML configuration file."""
    with open(path, 'r') as f:
        return yaml.safe_load(f)

def run_single_experiment(config_base, dim, net_type="EINet", epochs=20):
    """Run a single experiment with the given dimension and network type."""
    # Create a copy of the config to modify
    cfg = deepcopy(config_base)
    
    # Set the network type
    cfg["model"]["network"]["type"] = net_type
    
    # Set the branch factors based on dimension
    cfg["model"]["network"]["parameters"]["excitatory_branch_factors"] = [2, int(dim)]
    
    # Ensure no mismatch in inhibitory factors
    cfg["model"]["network"]["parameters"]["inhibitory_branch_factors"] = []
    
    # Set training parameters (fewer epochs for faster experiments)
    cfg["train"]["epochs"] = epochs
    
    # Create a temporary directory for outputs
    import tempfile
    temp_dir = tempfile.mkdtemp()
    config_path = os.path.join(temp_dir, "config.yaml")
    
    # Save the config to the temp directory
    with open(config_path, 'w') as f:
        yaml.dump(cfg, f)
    
    # Run the experiment
    exp_name = f"scaling_{net_type}_dim{dim}"
    logger.info(f"Running experiment: {exp_name}")
    
    try:
        train_main(config_path=config_path, output_dir=temp_dir, experiment_name=exp_name)
    except Exception as e:
        logger.error(f"Training failed: {e}")
        return None, None, None  # Return three None values when there's an error
    
    # Extract parameter count and loss/accuracy
    try:
        final_results_path = os.path.join(temp_dir, "results", f"{exp_name}_*", "final_results.json")
        import glob
        final_results_file = glob.glob(final_results_path)[0]
        
        import json
        with open(final_results_file, 'r') as f:
            results = json.load(f)
            
        param_count = results.get("param_count", 0)
        final_loss = results.get("final_loss", 0)
        
        # Alternative: load the model to count parameters and get performance
        model_path = glob.glob(os.path.join(temp_dir, "results", f"{exp_name}_*", "best_model.pt"))[0]
        model = torch.load(model_path)
        param_count = sum(p.numel() for p in model.parameters())
        
        # Get accuracy from performance file
        perf_path = glob.glob(os.path.join(temp_dir, "results", f"{exp_name}_*", "performance", "final.json"))[0]
        with open(perf_path, 'r') as f:
            perf_data = json.load(f)
        test_acc = perf_data.get("test accuracy", 0)
        
        return param_count, test_acc, final_loss
    
    except Exception as e:
        logger.error(f"Error extracting results: {e}")
        return None, None, None  # Return three None values for consistency
    finally:
        # Clean up
        import shutil
        shutil.rmtree(temp_dir)

In [ ]:
def run_scaling_experiment(base_config_path, dim_values, epochs=20):
    """Run scaling experiments for EINet and MLP with different dimensions."""
    # Load base configuration
    base_cfg = load_config_yaml(base_config_path)
    if not base_cfg:
        logger.error(f"Failed to load base config from {base_config_path}")
        return None
    
    # Store results in memory
    results = {
        "EINet": [],
        "MLP": []
    }
    
    # Run experiments for each dimension and network type
    for dim in dim_values:
        for net_type in ["EINet", "MLP"]:
            result = run_single_experiment(
                base_cfg, dim, net_type=net_type, epochs=epochs
            )
            
            # Unpack results - handle the case where we might get different number of values
            if isinstance(result, tuple):
                if len(result) == 3:
                    param_count, accuracy, loss = result
                elif len(result) == 2:
                    param_count, accuracy = result
                    loss = None
                else:
                    param_count = result[0] if result else None
                    accuracy = loss = None
            else:
                param_count = accuracy = loss = None
            
            if param_count is not None:
                results[net_type].append({
                    "dim": dim,
                    "param_count": param_count,
                    "accuracy": accuracy,
                    "loss": loss
                })
                logger.info(f"Completed {net_type} dim={dim}: params={param_count}, accuracy={accuracy}, loss={loss}")
            else:
                logger.warning(f"Failed to get results for {net_type} dim={dim}")
    
    return results

In [ ]:
# Define experiment parameters
BASE_CONFIG = "../.vscode/config_exp.yaml"  # Path to base configuration
DIM_LIST = [2, 4, 8, 16, 32]  # Dimensions to sweep
EPOCHS = 20  # Reduced epochs for faster runs

# Run the scaling experiments
results = run_scaling_experiment(BASE_CONFIG, DIM_LIST, epochs=EPOCHS)

In [ ]:
def plot_loglog(results):
    """Create log-log plots for parameter count vs loss and accuracy."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Configure log scales
    ax1.set_xscale("log")
    ax1.set_yscale("log")
    ax2.set_xscale("log")
    ax2.set_yscale("log")
    
    # Colors and markers for different network types
    styles = {
        "EINet": {"color": "blue", "marker": "o"},
        "MLP": {"color": "red", "marker": "s"}
    }
    
    # Plot each network type
    for net_type, style in styles.items():
        if net_type not in results:
            logger.warning(f"Network type {net_type} not found in results")
            continue
        
        # Get data and sort by parameter count
        data_list = results.get(net_type, [])
        data_list = sorted(data_list, key=lambda x: x["param_count"])
        
        if len(data_list) == 0:
            continue
            
        # Extract values for plotting
        x_param = [d["param_count"] for d in data_list]
        y_loss = [d["loss"] for d in data_list]
        y_error = [1.0 - d["accuracy"] for d in data_list]  # Convert accuracy to error rate
        dims = [d["dim"] for d in data_list]
        
        # Plot loss curve
        ax1.plot(x_param, y_loss, marker=style["marker"], color=style["color"], 
                 label=f"{net_type} (Loss)", linewidth=2, markersize=10)
        
        # Plot error rate curve
        ax2.plot(x_param, y_error, marker=style["marker"], color=style["color"], 
                linestyle='--', label=f"{net_type} (Error)", linewidth=2, markersize=10)
        
        # Add dimension labels
        for i, (x, y1, y2, dim) in enumerate(zip(x_param, y_loss, y_error, dims)):
            ax1.annotate(f"dim={dim}", xy=(x, y1), xytext=(10, 0),
                        textcoords='offset points', fontsize=10)
            ax2.annotate(f"dim={dim}", xy=(x, y2), xytext=(10, 0),
                        textcoords='offset points', fontsize=10)
    
    # Set labels and titles
    ax1.set_xlabel("Parameter Count (log scale)", fontsize=12)
    ax1.set_ylabel("Loss (log scale)", fontsize=12)
    ax1.set_title("Loss vs Parameter Count", fontsize=14)
    ax1.grid(True, which="both", alpha=0.3)
    ax1.legend(loc='best', fontsize=12)
    
    ax2.set_xlabel("Parameter Count (log scale)", fontsize=12)
    ax2.set_ylabel("Error Rate (log scale)", fontsize=12)
    ax2.set_title("Error Rate vs Parameter Count", fontsize=14)
    ax2.grid(True, which="both", alpha=0.3)
    ax2.legend(loc='best', fontsize=12)
    
    plt.tight_layout()
    return fig

# Plot the results
plot_loglog(results)
plt.show()

In [ ]:
def calculate_scaling_exponents(results):
    """Calculate the scaling exponents for loss and error rate vs parameter count."""
    for net_type in ["EINet", "MLP"]:
        data_list = results.get(net_type, [])
        data_list = sorted(data_list, key=lambda x: x["param_count"])
        
        if len(data_list) < 2:
            print(f"Not enough data points for {net_type} to calculate scaling exponent")
            continue
            
        # Extract log values
        log_params = np.log(np.array([d["param_count"] for d in data_list]))
        log_loss = np.log(np.array([d["loss"] for d in data_list]))
        log_error = np.log(np.array([1.0 - d["accuracy"] for d in data_list]))
        
        # Calculate scaling exponents using linear regression
        loss_slope, _ = np.polyfit(log_params, log_loss, 1)
        error_slope, _ = np.polyfit(log_params, log_error, 1)
        
        print(f"\n{net_type} Scaling:")
        print(f"  Loss scaling exponent: {loss_slope:.4f}")
        print(f"  Error scaling exponent: {error_slope:.4f}")
        
        if loss_slope < 0:
            print(f"  Loss scales with parameters as: loss ∝ (params)^{loss_slope:.4f}")
            print(f"  Doubling parameter count decreases loss by {2**abs(loss_slope)-1:.2%}")
        
        if error_slope < 0:
            print(f"  Error scales with parameters as: error ∝ (params)^{error_slope:.4f}")
            print(f"  Doubling parameter count decreases error by {2**abs(error_slope)-1:.2%}")

# Calculate scaling exponents
calculate_scaling_exponents(results)